In [1]:
# ============================================================
# NOTEBOOK 04 — RETRIEVAL
# ============================================================
# WHAT IS RETRIEVAL?
# We have a vector database full of embedded chunks.
# Now when a user asks a question we need to find
# the chunks that are most relevant to that question.
#
# How it works:
# 1. Convert the question into a vector (same embedding model)
# 2. Compare that vector against all stored chunk vectors
# 3. Return the top k most similar chunks
#
# This is called "semantic search" — we search by MEANING
# not by keywords. So even if the question uses different
# words than the document, we can still find relevant chunks.
#
# Example:
# Question : "How does the model pay attention?"
# Finds    : chunks about "attention mechanism"
# Even though the word "pay" isn't in the document!
# ============================================================

print("Starting retrieval notebook...")

Starting retrieval notebook...


In [2]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# HuggingFaceEmbeddings to reload our embedding model
from langchain_huggingface import HuggingFaceEmbeddings

# Chroma to connect to our existing vector database
from langchain_community.vectorstores import Chroma

import os

print("Libraries imported!")

Libraries imported!


In [3]:
# ============================================================
# STEP 1: CONNECT TO THE EXISTING VECTOR DATABASE
# ============================================================
# We don't need to re-embed everything!
# ChromaDB saved everything to disk in notebook 03.
# We just reconnect to it here.

# Reload the same embedding model
# IMPORTANT: must be the SAME model used in notebook 03
# Using a different model would give wrong results
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Connect to the existing ChromaDB database on disk
# We use the same path as notebook 03
vectorstore = Chroma(
    # Path where ChromaDB saved the database
    persist_directory="../.chroma",

    # Must be the same embedding model used to create it
    embedding_function=embedding_model
)

print(f"Connected to vector store!")
print(f"Total chunks in database: {vectorstore._collection.count()}")

/var/folders/d0/ymvkftpj5hzdbp0bfg1vxwvr0000gn/T/ipykernel_63271/2668646283.py:19: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


Connected to vector store!
Total chunks in database: 94


In [4]:
# ============================================================
# STEP 2: CREATE A RETRIEVER
# ============================================================
# A retriever is a simple interface that takes a question
# and returns the most relevant chunks.
# It wraps the vector store with a clean .invoke() method.

retriever = vectorstore.as_retriever(

    # "similarity" means find chunks most similar to the question
    # other options: "mmr" (diverse results), "similarity_score_threshold"
    search_type="similarity",

    search_kwargs={
        # k=3 means return the top 3 most relevant chunks
        # You can increase this for more context
        # but too many chunks can confuse the LLM
        "k": 3
    }
)

print("Retriever created!")
print("Search type : similarity")
print("Top k chunks: 3")

Retriever created!
Search type : similarity
Top k chunks: 3


In [5]:
# ============================================================
# STEP 3: TEST THE RETRIEVER WITH A QUESTION
# ============================================================

# Our test question
question = "What is the attention mechanism?"

# .invoke() searches the vector store and returns
# the top 3 most relevant Document chunks
retrieved_chunks = retriever.invoke(question)

print(f"Question: {question}")
print(f"Number of chunks retrieved: {len(retrieved_chunks)}")
print()

# Look at each retrieved chunk
for i, chunk in enumerate(retrieved_chunks):
    print(f"{'='*50}")
    print(f"CHUNK {i+1}")
    print(f"{'='*50}")
    print(f"Page   : {chunk.metadata.get('page', 'unknown')}")
    print(f"Content: {chunk.page_content}")
    print()

Question: What is the attention mechanism?
Number of chunks retrieved: 3

CHUNK 1
Page   : 12
Content: .
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for
the word ‘making’. Different colors represent different heads. Best viewed in color.
13

CHUNK 2
Page   : 14
Content: but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the
sentence. We give two such examples above, from two different heads from the encoder self-attention
at layer 5 of 6. The heads clearly learned to perform different tasks.
15

CHUNK 3
Page   : 1
Content: reduced to a constant number of operations, a

In [6]:
# ============================================================
# STEP 4: TRY DIFFERENT QUESTIONS
# ============================================================
# Let's see how the retriever handles different questions
# This shows the power of semantic search

questions = [
    "How does multi-head attention work?",
    "What optimizer was used for training?",
    "What are the results on translation tasks?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print(f"{'='*60}")

    # Get relevant chunks for this question
    chunks = retriever.invoke(question)

    # Show just the first chunk for each question
    print(f"Most relevant chunk (page {chunks[0].metadata.get('page', '?')}):")
    print(chunks[0].page_content[:400])


QUESTION: How does multi-head attention work?
Most relevant chunk (page 1):
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. 

QUESTION: What optimizer was used for training?
Most relevant chunk (page 6):
(3.5 days).
5.3 Optimizer
We used the Adam optimizer [20] with β1 = 0.9, β2 = 0.98 and ϵ = 10−9. We varied the learning
rate over the course of training, according to the formula:
lrate = d−0.5model · min(step_num−0.5, step_num · warmup_steps−1.5) (3)
This corresponds to increasing the learning rate linearly for the first warmup_steps training steps,
and decreasing it thereafter proportionally to 

QUESTION: What are the results on transla

In [7]:
# ============================================================
# STEP 5: GET SIMILARITY SCORES WITH RESULTS
# ============================================================
# similarity_search_with_score returns the chunks
# AND their similarity scores so we can see HOW relevant
# each chunk is to the question

question = "What is the attention mechanism?"

# Returns list of (Document, score) tuples
# Lower score = more similar (it's a distance score)
results_with_scores = vectorstore.similarity_search_with_score(
    question,
    k=3
)

print(f"Question: {question}\n")

for i, (chunk, score) in enumerate(results_with_scores):
    print(f"--- RESULT {i+1} ---")
    print(f"Similarity score : {score:.4f}  (lower = more relevant)")
    print(f"Page             : {chunk.metadata.get('page', '?')}")
    print(f"Content          : {chunk.page_content[:250]}")
    print()

Question: What is the attention mechanism?

--- RESULT 1 ---
Similarity score : 0.6560  (lower = more relevant)
Page             : 12
Content          : .
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘ma

--- RESULT 2 ---
Similarity score : 0.7191  (lower = more relevant)
Page             : 14
Content          : but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the
sentence. We give two such examples above, from two diff

--- RESULT 3 ---
Similarity score : 0.7948  (lower = more relevant)
Page             : 1
Content          : reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weig

In [8]:
# ============================================================
# SUMMARY — WHAT DID WE LEARN?
# ============================================================

print("=== RETRIEVAL SUMMARY ===")
print("Retriever type  : similarity search")
print("Embedding model : sentence-transformers/all-MiniLM-L6-v2")
print("Chunks returned : top 3 per question")

print("""
WHAT HAPPENS NEXT?
------------------
We can now retrieve the most relevant chunks
for any question automatically.

The next step is PROMPTING & GENERATION.

We take the retrieved chunks and combine them
with the user question into a PROMPT that we
send to the LLM.

The LLM then generates an answer based ONLY
on those chunks — this is what makes RAG
accurate and grounded in your documents!

See notebook 05_prompting_generation.ipynb!
""")

=== RETRIEVAL SUMMARY ===
Retriever type  : similarity search
Embedding model : sentence-transformers/all-MiniLM-L6-v2
Chunks returned : top 3 per question

WHAT HAPPENS NEXT?
------------------
We can now retrieve the most relevant chunks
for any question automatically.

The next step is PROMPTING & GENERATION.

We take the retrieved chunks and combine them
with the user question into a PROMPT that we
send to the LLM.

The LLM then generates an answer based ONLY
on those chunks — this is what makes RAG
accurate and grounded in your documents!

See notebook 05_prompting_generation.ipynb!

